In [ ]:
# 🛠️ 1. OpenAI 0.28 버전 설치
!pip uninstall -y openai
!pip install openai==0.28

# 📦 2. 라이브러리 로드
import openai
import pandas as pd
import time
from google.colab import files

Found existing installation: openai 0.28.0
Uninstalling openai-0.28.0:
  Successfully uninstalled openai-0.28.0
  Using cached openai-0.28.0-py3-none-any.whl.metadata (13 kB)
Using cached openai-0.28.0-py3-none-any.whl (76 kB)


In [ ]:
# 🔑 3. OpenAI API 키 입력
openai.api_key = "-"  # ← 여기에 본인 API 키 넣기

# 📂 4. CSV 업로드
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])
texts = df["text"].dropna().tolist()[1100:1460]  # 상위 100개만

Saving data_.csv to data_ (17).csv


In [ ]:
# ⚙️ 4. 시스템 프롬프트 설정
system_prompt = """다음 규칙에 따라 고객의 발화를 정중하게 순화하세요:
1. 욕설, 혐오, 무례한 표현이 없다면 그대로 출력하세요.
2. 욕설, 혐오, 무례한 표현만 있는 경우에는 '강한 불쾌감 표현' 또는 '응대 방식에 대한 불쾌감 표현'이라고 출력하세요.
3. 욕설·무례함과 함께 실질적 요청이 있는 경우에는 정중한 고객 발화로 순화하세요.
4. 단순히 말투만 바꾸는 것이 아니라, 상담사가 감정적으로 상처받지 않도록 표현을 바꾸어야 합니다.
5. 반드시 '고객의 말투'로 순화하세요.
   ✅ 상담사의 사과나 안내 문구로 바꾸지 마세요.
"""


# 💬 5. Few-shot 예시 5개 (토큰 절약용)
few_shots = [
    {"role": "user", "content": "야 이 좆같은 상담 꺼져라"},
    {"role": "assistant", "content": "강한 불쾌감 표현"},

    {"role": "user", "content": "왜 이렇게 느려요 씨발"},
    {"role": "assistant", "content": "왜 이렇게 느린 건가요? 조금 더 빠르게 처리해 주세요."},

    {"role": "user", "content": "이런 상담 받을 거면 걍 죽는 게 낫겠다"},
    {"role": "assistant", "content": "응대 방식에 대해 강한 불쾌감을 표현하고 있습니다."}
]


# 🚀 6. Batch 실행
import time

def sanitize_batch(batch):
    joined_input = "\n".join([f"{i+1}. {s}" for i, s in enumerate(batch)])
    user_prompt = f"""다음 문장들을 순화해 주세요. 각 문장별로 줄마다 순서대로 출력해 주세요. 단, 번호는 붙이지 말고 문장만 출력해 주세요:\n{joined_input}"""

    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[{"role": "system", "content": system_prompt}] + few_shots + [{"role": "user", "content": user_prompt}],
            temperature=0.3,
            max_tokens=400
        )
        result = response.choices[0].message.content.strip()
        return result.split("\n")
    except Exception as e:
        print(f"❌ 에러 발생: {e}")
        time.sleep(3)
        return ["[ERROR]"] * len(batch)

# 🎯 7. 실제 실행
results = []
batch_size = 5

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    outputs = sanitize_batch(batch)
    results.extend(outputs)
    print(f"✅ {i+1}~{i+len(batch)} 변환 완료")
    time.sleep(1.5)  # 속도 제한 방지용

# 💾 8. 결과 저장
df_result = pd.DataFrame({"원문": texts, "순화된_표현": results})
df_result.to_csv("순화_결과_100개.csv", index=False)
print("📁 변환 완료 → '순화_결과_100개.csv' 저장됨")


✅ 1~5 변환 완료
✅ 6~10 변환 완료
✅ 11~15 변환 완료
✅ 16~20 변환 완료
✅ 21~25 변환 완료
✅ 26~30 변환 완료
✅ 31~35 변환 완료
✅ 36~40 변환 완료
✅ 41~45 변환 완료
✅ 46~50 변환 완료
✅ 51~55 변환 완료
✅ 56~60 변환 완료
✅ 61~65 변환 완료
✅ 66~70 변환 완료
✅ 71~75 변환 완료
✅ 76~80 변환 완료
✅ 81~85 변환 완료
✅ 86~90 변환 완료
✅ 91~95 변환 완료
✅ 96~100 변환 완료
✅ 101~105 변환 완료
✅ 106~110 변환 완료
✅ 111~115 변환 완료
✅ 116~120 변환 완료
✅ 121~125 변환 완료
✅ 126~130 변환 완료
✅ 131~135 변환 완료
✅ 136~140 변환 완료
✅ 141~145 변환 완료
✅ 146~150 변환 완료
✅ 151~155 변환 완료
✅ 156~160 변환 완료
✅ 161~165 변환 완료
✅ 166~170 변환 완료
✅ 171~175 변환 완료
✅ 176~180 변환 완료
✅ 181~185 변환 완료
✅ 186~190 변환 완료
✅ 191~195 변환 완료
✅ 196~200 변환 완료
✅ 201~205 변환 완료
✅ 206~210 변환 완료
✅ 211~215 변환 완료
✅ 216~220 변환 완료
✅ 221~225 변환 완료
✅ 226~230 변환 완료
✅ 231~235 변환 완료
✅ 236~240 변환 완료
✅ 241~245 변환 완료
✅ 246~250 변환 완료
✅ 251~255 변환 완료
✅ 256~260 변환 완료
✅ 261~265 변환 완료
✅ 266~270 변환 완료
✅ 271~275 변환 완료
✅ 276~280 변환 완료
✅ 281~285 변환 완료
✅ 286~290 변환 완료
✅ 291~295 변환 완료
✅ 296~300 변환 완료
✅ 301~305 변환 완료
✅ 306~310 변환 완료
✅ 311~315 변환 완료
✅ 316~320 변환 완료
✅ 321~325 변환 완료
✅ 

In [ ]:
from google.colab import files

# 💾 CSV 저장
df_result.to_csv("순화_결과_100개.csv", index=False)

# 📥 다운로드
files.download("순화_결과_100개.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>